# < K-S Test>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

In [4]:
from sklearn.datasets import load_iris

iris = load_iris()

X = pd.DataFrame(iris["data"],
                 columns=iris["feature_names"])

y = pd.Series(iris["target"],
               name="target")
data = pd.concat([X, y], axis=1)
data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [5]:
cond = (data["target"] == 0)
cond1 = (data["target"] == 1)
data1 = data.loc[cond, "sepal length (cm)"].to_numpy()
data2 = data.loc[cond1, "sepal length (cm)"].to_numpy()
data1

array([5.1, 4.9, 4.7, 4.6, 5. , 5.4, 4.6, 5. , 4.4, 4.9, 5.4, 4.8, 4.8,
       4.3, 5.8, 5.7, 5.4, 5.1, 5.7, 5.1, 5.4, 5.1, 4.6, 5.1, 4.8, 5. ,
       5. , 5.2, 5.2, 4.7, 4.8, 5.4, 5.2, 5.5, 4.9, 5. , 5.5, 4.9, 4.4,
       5.1, 5. , 4.5, 4.4, 5. , 5.1, 4.8, 5.1, 4.6, 5.3, 5. ])

In [6]:
data2

array([7. , 6.4, 6.9, 5.5, 6.5, 5.7, 6.3, 4.9, 6.6, 5.2, 5. , 5.9, 6. ,
       6.1, 5.6, 6.7, 5.6, 5.8, 6.2, 5.6, 5.9, 6.1, 6.3, 6.1, 6.4, 6.6,
       6.8, 6.7, 6. , 5.7, 5.5, 5.5, 5.8, 6. , 5.4, 6. , 6.7, 6.3, 5.6,
       5.5, 5.5, 6.1, 5.8, 5. , 5.6, 5.7, 5.7, 6.2, 5.1, 5.7])

In [7]:
### 1표본 K-S 검정
from scipy.stats import kstest

# H0 : 데이터는 정규분포와 분포가 같다.
# H1 : 데이터는 정규분포와 분포가 다르다.

m = data1.mean()
std = data1.std(ddof=1)

s, p = kstest(data1,            ## 원본 데이터 
              "norm",           #  분포 형태
              args=(m,std))    #  표본에서 추정한 모수값 넣기

print(f"검정통계량 : {s}")
print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량 : 0.11485990669608126
검정통계량의 p-value : 0.4889236515009082
귀무가설 채택


In [8]:
### 2표본 K-S 검정
from scipy.stats import ks_2samp

# H0 : 두 집단의 분포가 서로 같다.
# H1 : 두 집단의 분포가 서로 다르다.

s, p = ks_2samp(data1, data2, 
                alternative="two-sided")

print(f"검정통계량 : {s}")
print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량 : 0.78
검정통계량의 p-value : 2.807570962237254e-15
귀무가설 기각


In [9]:
print("""
유의수준 5%하에서 귀무가설을 기각한다. 즉, 두 집단의 분포는 서로 다르다.
""")


유의수준 5%하에서 귀무가설을 기각한다. 즉, 두 집단의 분포는 서로 다르다.



## <33회 기출문제>

In [138]:
df = pd.read_csv('https://raw.githubusercontent.com/Datamanim/datarepo/refs/heads/master/adp/33/data/s2.csv')
df

,지연ID,지연일자,노선,최대지연시간
0,ID_0,2023-09-01,1호선,15분
1,ID_1,2023-09-01,1호선,15분
2,ID_2,2023-09-01,4호선,10분
3,ID_3,2023-09-01,4호선,10분
4,ID_4,2023-09-04,4호선,10분
...,...,...,...,...
828,ID_828,2024-08-28,4호선,15분
829,ID_829,2024-08-29,4호선,10분
830,ID_830,2024-08-29,4호선,10분
831,ID_831,2024-08-30,4호선,10분


In [139]:
### EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   지연ID    833 non-null    object
 1   지연일자    833 non-null    object
 2   노선      833 non-null    object
 3   최대지연시간  833 non-null    object
dtypes: object(4)
memory usage: 26.2+ KB


In [140]:
### 결측치 확인
df.isnull().sum()

지연ID      0
지연일자      0
노선        0
최대지연시간    0
dtype: int64

## 6-1 노선에 상관없이 일별 최대 지연시간이 5~15분으로 발생하는 경우는 하나의 사건으로 보자. 해당 사건이 일자별 발생하는 빈도가 푸아송분포를 따르는지 확인하는 방법 2가지를 기술하고 결과를 보여라.

In [141]:
df["최대지연시간"].value_counts()

10분       414
5분        179
15분       153
20분        58
25분        16
30분 이상     13
Name: 최대지연시간, dtype: int64

In [142]:
df1 = df.copy()

In [143]:
def sep_time(x):
    if (x == "5분") | (x == "10분") | (x == "15분"):
        return "5~15분"
    elif x == "20분":
        return "20분"
    elif x == "25분":
        return "25분"
    elif x == "30분 이상":
        return "30분 이상"

df1["최대지연시간"] = df1["최대지연시간"].apply(sep_time)
df1["최대지연시간"].value_counts()

5~15분     746
20분        58
25분        16
30분 이상     13
Name: 최대지연시간, dtype: int64

In [144]:
cond = (df1["최대지연시간"] == "5~15분")
df_temp = df1.loc[cond, :].copy()
df_temp

,지연ID,지연일자,노선,최대지연시간
0,ID_0,2023-09-01,1호선,5~15분
1,ID_1,2023-09-01,1호선,5~15분
2,ID_2,2023-09-01,4호선,5~15분
3,ID_3,2023-09-01,4호선,5~15분
4,ID_4,2023-09-04,4호선,5~15분
...,...,...,...,...
828,ID_828,2024-08-28,4호선,5~15분
829,ID_829,2024-08-29,4호선,5~15분
830,ID_830,2024-08-29,4호선,5~15분
831,ID_831,2024-08-30,4호선,5~15분


In [145]:
df_temp = df_temp.groupby(["지연일자","최대지연시간"]).size().reset_index().copy()
df_temp

,지연일자,최대지연시간,0
0,2023-09-01,5~15분,4
1,2023-09-04,5~15분,2
2,2023-09-05,5~15분,3
3,2023-09-06,5~15분,4
4,2023-09-08,5~15분,4
...,...,...,...
250,2024-08-26,5~15분,2
251,2024-08-27,5~15분,2
252,2024-08-28,5~15분,2
253,2024-08-29,5~15분,2


In [146]:
### 기간내 모든 날짜 만들기
all_dates = pd.date_range(start="2023-09-01",      ## 시작 날짜
                          end="2024-08-31"         ## 마지막 날짜
                          )
all_dates

DatetimeIndex(['2023-09-01', '2023-09-02', '2023-09-03', '2023-09-04',
               '2023-09-05', '2023-09-06', '2023-09-07', '2023-09-08',
               '2023-09-09', '2023-09-10',
               ...
               '2024-08-22', '2024-08-23', '2024-08-24', '2024-08-25',
               '2024-08-26', '2024-08-27', '2024-08-28', '2024-08-29',
               '2024-08-30', '2024-08-31'],
              dtype='datetime64[ns]', length=366, freq='D')

In [147]:
df_temp = df_temp.set_index("지연일자").copy()
df_temp

,최대지연시간,0
지연일자,,
2023-09-01,5~15분,4
2023-09-04,5~15분,2
2023-09-05,5~15분,3
2023-09-06,5~15분,4
2023-09-08,5~15분,4
...,...,...
2024-08-26,5~15분,2
2024-08-27,5~15분,2
2024-08-28,5~15분,2


In [148]:
### 새로운 인덱스로 재설정

df_temp = df_temp.reindex(all_dates.strftime("%Y-%m-%d"),       ## 새로운 인덱스값   
                          fill_value=0)                         #  기존에 없던 인덱스에 채워넣을 값 

cond = (df_temp["최대지연시간"] == 0)
df_temp.loc[cond, "최대지연시간"] = "없음"
df_temp

,최대지연시간,0
2023-09-01,5~15분,4
2023-09-02,없음,0
2023-09-03,없음,0
2023-09-04,5~15분,2
2023-09-05,5~15분,3
...,...,...
2024-08-27,5~15분,2
2024-08-28,5~15분,2
2024-08-29,5~15분,2
2024-08-30,5~15분,2


In [149]:
data_group = df_temp[0].to_numpy()
data_group

array([4, 0, 0, 2, 3, 4, 0, 4, 4, 0, 4, 4, 2, 4, 1, 0, 1, 4, 2, 2, 4, 3,
       2, 0, 2, 4, 0, 0, 2, 0, 0, 0, 1, 4, 0, 2, 1, 0, 0, 2, 4, 2, 0, 2,
       1, 4, 0, 4, 2, 4, 0, 0, 2, 3, 4, 4, 4, 0, 0, 2, 4, 4, 0, 4, 0, 0,
       2, 3, 3, 3, 6, 0, 1, 2, 0, 4, 4, 5, 0, 0, 4, 4, 2, 2, 3, 0, 0, 2,
       1, 4, 1, 3, 0, 0, 0, 4, 2, 3, 2, 2, 0, 3, 2, 4, 2, 4, 0, 0, 2, 4,
       4, 2, 2, 0, 0, 0, 2, 3, 2, 2, 0, 0, 0, 4, 4, 2, 2, 0, 0, 3, 2, 1,
       2, 6, 0, 2, 2, 4, 2, 4, 2, 0, 0, 1, 1, 2, 3, 4, 0, 0, 0, 2, 2, 4,
       1, 0, 0, 2, 4, 2, 2, 0, 0, 0, 0, 4, 0, 3, 2, 0, 0, 4, 4, 4, 1, 2,
       2, 0, 0, 4, 3, 4, 0, 0, 0, 4, 0, 2, 2, 4, 0, 0, 2, 4, 2, 4, 2, 0,
       0, 4, 2, 4, 2, 2, 0, 0, 6, 3, 2, 6, 3, 2, 0, 3, 4, 5, 6, 7, 4, 0,
       2, 8, 2, 4, 2, 5, 0, 3, 4, 7, 6, 4, 0, 2, 4, 1, 4, 6, 2, 0, 1, 4,
       4, 0, 2, 6, 1, 0, 2, 6, 5, 3, 2, 4, 0, 2, 2, 2, 2, 4, 2, 2, 2, 1,
       2, 4, 5, 2, 0, 6, 0, 4, 2, 4, 0, 2, 3, 4, 2, 0, 2, 0, 0, 4, 1, 4,
       2, 0, 0, 0, 2, 5, 2, 2, 3, 0, 2, 4, 2, 2, 2,

In [150]:
### 1. 포아송 분포는 평균과 분산이 같으므로 이것을 확인하여 포아송 분포인지 아닌지 검정한다.

m = data_group.mean()
v = data_group.var(ddof=1)

print(f"평균 : {m}")
print(f"분산 : {v}")

평균 : 2.0382513661202184
분산 : 3.039628714724156


In [151]:
print("""
평균과 분산이 일치하지않고 평균 < 분산 이므로 과산포로 음이항분포를 따름을 확인하였다.
""")


평균과 분산이 일치하지않고 평균 < 분산 이므로 과산포로 음이항분포를 따름을 확인하였다.



In [152]:
### 1 sample K-S test
from scipy.stats import kstest

# H0 : 데이터 분포가 포아송 분포와 같다.
# H1 : 데이터 분포가 포아송 분포와 다르다.

mu = data_group.mean()

s, p = kstest(data_group,
              "poisson",
              args=(mu,))

print(f"검정통계량 : {s}")
print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량 : 0.3084006503187105
검정통계량의 p-value : 2.082463914377942e-31
귀무가설 기각


In [153]:
print("""
유의수준 5%하에서 귀무가설을 기각한다. 즉, 데이터의 분포가 포아송 분포와 다르다.
""")


유의수준 5%하에서 귀무가설을 기각한다. 즉, 데이터의 분포가 포아송 분포와 다르다.

